# Querying for RAG

This notebook covers the basic query steps for a RAG system
- retrieve nodes from the index
- post-process retrieved nodes
  - e.g., re-ranking
- call an LLM to synthesize a query response

In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

## Let's start by configuring Llamaindex

Here is how you tell Llamaindex which embed model and llm model you want to use for your project.

You can override these models for individual components, but the models you set in the global `Settings` object will be the defaults

In [2]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

In [3]:
embed_model = OpenAIEmbedding(model="text-embedding-3-small", embed_batch_size=100)
llm = OpenAI(model="gpt-3.5-turbo", temperature=0.0)

Settings.embed_model = embed_model
Settings.llm = llm

## Next we will create a simple index

So we have something to query!

In [4]:
import logging
import os
import sys

from llama_index.core import Document
from llama_index.core.node_parser import MarkdownNodeParser

In [5]:
# configure
filename = 'sleeping_gods.md'

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [6]:
with open(f'data/{filename}', 'r', encoding='utf-8') as file:
    document = Document(
        text = file.read(),
        metadata = {"filename": filename},
    )
print(len(document.text))

77034


In [7]:
splitter = MarkdownNodeParser()
nodes = splitter.get_nodes_from_documents([document], show_progress=False)
print(len(nodes))

97


In [8]:
for ix, node in enumerate(nodes[0:3]):
    print(f">>>{ix} {node.id_}")
    print("Metadata", node.metadata)
    print("Text", node.text[:200])
    print("\n\n")

>>>0 803ab030-1775-45d4-906d-0c79afee321c
Metadata {'Header_2': 'Overview (page 1)', 'filename': 'sleeping_gods.md'}
Text Overview (page 1)

1-4 players, ages 13+, 1-20 hours

"This is the Wandering Sea. The gods have brought you here, and you must wake them if you wish to return home."

In Sleeping Gods, you and up to t



>>>1 59eb3a6f-9f97-4c9a-98b2-47b6d15d7682
Metadata {'Header_2': 'Setup (page 4)', 'filename': 'sleeping_gods.md'}
Text Setup (page 4)

Follow these instructions if you are starting a new campaign. If this is your first campaign, we recommend using the quick start guide first. If you are setting up the game to continue



>>>2 1c4e343b-83dc-4d99-89e0-b0e1a7d08cf7
Metadata {'Header_2': 'Basics (page 6)', 'filename': 'sleeping_gods.md'}
Text Basics (page 6)

Pgs. 6-9 introduce some basics of Sleeping Gods to help you get your sea legs. Turn structure and actions are explained starting on pg. 10.





In [9]:
import chromadb

from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.vector_stores.types import MetadataInfo, VectorStoreInfo
from llama_index.vector_stores.chroma import ChromaVectorStore

In [10]:
# create an ephemeral (non-persistent) chroma collection
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("test")

# create a vector store from the chroma collection
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# create the index from the nodes, storing the results in the vector store
index = VectorStoreIndex(
    nodes, 
    embed_model=embed_model,  # this is where we can override the embed model in the global Settings object if we wanted to
    storage_context=storage_context,
)

# create a retriever from the index
retriever = VectorIndexRetriever(
    index,
    similarity_top_k=3,
)

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


### Test the retriever

In [11]:
result = retriever.retrieve("What actions can I take during my turn?")
print(len(result))
for node in result:
    print(node)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
3
Node ID: 921b0929-be20-43ee-b70c-edaf9c2af2ce
Text: Turn Overview (page 10)  This is a summary of the steps you take
on your turn. The steps are explained in detail on pgs. 10-16.
Starting with the first player, players take turns in clockwise order.
Follow these steps on your turn:  1. Ship Action: Choose a ship
action. Move the ship action figure to one of the five ship rooms and
apply the eff...
Score:  0.437

Node ID: 4c496bfb-bc54-4dcc-9383-660ff9d1cc99
Text: 3. Two Actions (page 13)  Perform 2 of the following actions:
travel, explore, market, or port.  - You may choose the same action
twice.  - You may choose to skip an action to gain 1 command. You may
do this for one or both actions.
Score:  0.418

Node ID: 5d2e11c3-6f99-4647-b6ce-b182b0f38055
Text: 1. Ship Action (page 10)  Move the ship action figure to one of
the fi

## Let's query the index

There are two ways to query the index
- the easy way
- the hard way

You get more control doing it the hard way!

### First, the easy way

In [12]:
# create a query engine from the index
query_engine = index.as_query_engine()

# now query the query engine
response = query_engine.query("What actions can I take during my turn?")
print(response)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
During your turn, you can choose to perform two actions from the following options: travel, explore, market, or port. Additionally, you have the choice to skip an action to gain 1 command, and you can do this for one or both of your actions.


### Now for the hard way

Note: It's not that hard

In [13]:
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers.type import ResponseMode
from llama_index.core.prompts.default_prompt_selectors import (
    DEFAULT_REFINE_PROMPT_SEL,
    DEFAULT_TEXT_QA_PROMPT_SEL,
    DEFAULT_TREE_SUMMARIZE_PROMPT_SEL,
)
from llama_index.core.prompts.default_prompts import DEFAULT_SIMPLE_INPUT_PROMPT

In [14]:
print("\nDEFAULT_TEXT_QA_PROMPT_SEL", DEFAULT_TEXT_QA_PROMPT_SEL)


DEFAULT_TEXT_QA_PROMPT_SEL metadata={'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>} template_vars=['context_str', 'query_str'] kwargs={} output_parser=None template_var_mappings={} function_mappings={} default_template=PromptTemplate(metadata={'prompt_type': <PromptType.QUESTION_ANSWER: 'text_qa'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings=None, function_mappings=None, template='Context information is below.\n---------------------\n{context_str}\n---------------------\nGiven the context information and not prior knowledge, answer the query.\nQuery: {query_str}\nAnswer: ') conditionals=[(<function is_chat_model at 0x12a41b740>, ChatPromptTemplate(metadata={'prompt_type': <PromptType.CUSTOM: 'custom'>}, template_vars=['context_str', 'query_str'], kwargs={}, output_parser=None, template_var_mappings=None, function_mappings=None, message_templates=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, content="You are an expert Q

In [15]:
# configure response synthesizer
# You can just call get_response_synthesizer() to get a response synthesizer.
# But below I'm setting the parameters to their default values just so you can see what parameters are available
response_synthesizer = get_response_synthesizer(
    llm = Settings.llm,  # this is where we could override the llm in the global Settings object if we wanted to
    text_qa_template = DEFAULT_TEXT_QA_PROMPT_SEL,
    # these other prompts are special-purpose; you probably won't use them
    refine_template = DEFAULT_REFINE_PROMPT_SEL,
    simple_template = DEFAULT_SIMPLE_INPUT_PROMPT,
    summary_template = DEFAULT_TREE_SUMMARIZE_PROMPT_SEL,
    # response mode is also special-purpose; leave it as compact
    response_mode = ResponseMode.COMPACT,
    callback_manager = Settings.callback_manager,
    use_async = False,
    streaming = False,
    structured_answer_filtering = False,
    verbose = False,
)

In [16]:
# configure node post-processors
# these could re-rank nodes, augment node text, or filter nodes as we are doing below
filter_less_similar_nodes = SimilarityPostprocessor(similarity_cutoff=0.3)
node_postprocessors = [filter_less_similar_nodes]

In [17]:
# create the query engine from the retriever, node post-processors, and the response synthesizer
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    node_postprocessors=node_postprocessors,
    response_synthesizer=response_synthesizer,
)

# query
response = query_engine.query("What actions can I take during my turn?")
print(response)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
During your turn, you can choose a ship action, draw an event card and resolve its effect, and then perform two actions from the following options: travel, explore, market, or port. You have the flexibility to choose the same action twice or skip an action to gain 1 command.
